Import Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
import pandas as pd
import pickle
from collections import defaultdict
import math


In [2]:
def recall_at_k(y_true, y_score, k=10):
    """
    Compute Recall@K for multi-label drug recommendation.
    y_true: (N, D) binary multi-hot
    y_score: (N, >=D) predicted scores (may include padding)
    """
    recalls = []
    for yt, ys in zip(y_true, y_score):
        D = yt.shape[0]          # true number of drugs
        ys = ys[:D]              #  truncate padded scores

        topk_idx = np.argsort(ys)[::-1][:min(k, D)]
        hits = yt[topk_idx].sum()
        recall = hits / yt.sum() if yt.sum() > 0 else 0.0
        recalls.append(recall)

    return np.array(recalls)


In [3]:
def ndcg_at_k(y_true, y_score, k=10):
    """
    NDCG@K for multi-label recommendation.
    y_true: (N, D) multi-hot
    y_score: (N, D) predicted scores
    """
    ndcgs = []

    for yt, ys in zip(y_true, y_score):
        D = yt.shape[0]
        ys = ys[:D]

        k_eff = min(k, D)

        # Top-k indices
        ranked_idx = np.argsort(-ys)[:k_eff]

        # DCG
        gains = yt[ranked_idx]
        discounts = 1.0 / np.log2(np.arange(2, k_eff + 2))
        dcg = np.sum(gains * discounts)

        # IDCG (ideal ranking)
        ideal_gains = np.sort(yt)[::-1][:k_eff]
        idcg = np.sum(ideal_gains * discounts)

        if idcg == 0:
            ndcgs.append(0.0)
        else:
            ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)


In [4]:
def ddi_rate (y_score, ddi_matrix, top_k=10):
    """
    Recommended-set DDI@K 

    y_score: (N, D) predicted scores
    ddi_matrix: (D, D) severity or binary DDI matrix
    """
    ddi_count = 0
    pair_count = 0
    severity_sum = 0.0

    for ys in y_score:
        D = ddi_matrix.shape[0]
        ys = ys[:D]  # truncate padding if any

        topk_idx = np.argsort(ys)[::-1][:min(top_k, D)]

        for i, d1 in enumerate(topk_idx):
            for d2 in topk_idx[i + 1:]:
                pair_count += 1
                sev = ddi_matrix[d1, d2]
                if sev > 0:
                    ddi_count += 1
                    severity_sum += sev

    ddi_rate_val = ddi_count / pair_count if pair_count > 0 else 0.0
    avg_severity = severity_sum / ddi_count if ddi_count > 0 else 0.0

    return ddi_rate_val, avg_severity


Data Preprocessing

In [5]:
# ============================================================
#  COMPLETE PIPELINE: EXPANDED CANCER COVERAGE
# WITHOUT CLASS WEIGHTS
# ============================================================

import numpy as np
import pandas as pd
import pickle
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

print("\n" + "="*70)
print("EXPANDED CANCER CLASSIFICATION + ABLATION STUDY")
print("="*70)

# =========================================================
# ABLATION STUDY CONFIGURATION
# =========================================================
USE_RANDOM_LAB = False       
USE_RANDOM_RAD = False       
USE_RANDOM_DRUG_DESC = False  
USE_HIERARCHICAL = True       
RANDOM_SEED = 42

# =========================================================
# PATHS
# =========================================================
EMB_RAD_PATH = r"....cancer_admission_embs_radiology.npy"
EMB_LAB_PATH = r"....solid_cancer_lab_embs.npy"
DRUG_EMB_PATH = r"...cancer_admission_embs_drugs.npy"
DRUG_SEQ_PATH = r"....cancer_drug_sequences.npy"
DDI_PATH = r"....mapped_ddi_pairs.pkl"
DIAGNOSES_PATH = r".....mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"
DRUG2IDX_PATH = r".....drug2idx.pkl"

# =========================================================
# STEP 1: EXPANDED ICD CANCER CODE LISTS
# =========================================================
print("\n" + "-"*40)
print("DEFINING EXPANDED CANCER TYPE MAPPING...")

# Complete ICD-9 Cancer Codes (ALL solid tumors)
EXPANDED_ICD9 = (
    # Head & Neck (140-149)
    "140","141","142","143","144","145","146","147","148","149",
    # Digestive (150-159)
    "150","151","152","153","154","155","156","157","158","159",
    # Respiratory (160-165)
    "160","161","162","163","164","165",
    # Bone/Soft Tissue (170-176)
    "170","171","172","173","174","175","176",
    # Genitourinary (179-189)
    "179","180","181","182","183","184","185","186","187","188","189",
    # Brain/CNS (190-192)
    "190","191","192",
    # Thyroid/Endocrine (193-199)
    "193","194","195","196","197","198","199"
)

# Complete ICD-10 Cancer Codes (ALL solid tumors)
EXPANDED_ICD10 = (
    # Head & Neck (C00-C14)
    "C00","C01","C02","C03","C04","C05","C06","C07","C08","C09","C10","C11","C12","C13","C14",
    # Digestive (C15-C26)
    "C15","C16","C17","C18","C19","C20","C21","C22","C23","C24","C25","C26",
    # Respiratory (C30-C39)
    "C30","C31","C32","C33","C34","C37","C38","C39",
    # Bone/Soft Tissue (C40-C49)
    "C40","C41","C43","C44","C45","C46","C47","C48","C49",
    # Breast (C50)
    "C50",
    # Female Genital (C51-C58)
    "C51","C52","C53","C54","C55","C56","C57","C58",
    # Male Genital (C60-C63)
    "C60","C61","C62","C63",
    # Urinary (C64-C68)
    "C64","C65","C66","C67","C68",
    # Brain/CNS (C69-C72)
    "C69","C70","C71","C72",
    # Thyroid/Endocrine (C73-C75)
    "C73","C74","C75"
)

print(f" Expanded ICD-9 prefixes: {len(EXPANDED_ICD9)}")
print(f" Expanded ICD-10 prefixes: {len(EXPANDED_ICD10)}")

# =========================================================
# STEP 2: COMPREHENSIVE ICD TO CANCER TYPE MAPPING
# =========================================================

# Complete ICD-9 to Cancer Type Mapping
ICD9_CANCER_MAPPING = {
    # Head and Neck (140-149)
    '140': 'HEAD_NECK_CANCER', '141': 'HEAD_NECK_CANCER', '142': 'HEAD_NECK_CANCER',
    '143': 'HEAD_NECK_CANCER', '144': 'HEAD_NECK_CANCER', '145': 'HEAD_NECK_CANCER',
    '146': 'HEAD_NECK_CANCER', '147': 'HEAD_NECK_CANCER', '148': 'HEAD_NECK_CANCER',
    '149': 'HEAD_NECK_CANCER',
    
    # Digestive System
    '150': 'ESOPHAGEAL_CANCER', '151': 'STOMACH_CANCER',
    '152': 'SMALL_INTESTINE_CANCER', '153': 'COLORECTAL_CANCER', '154': 'COLORECTAL_CANCER',
    '155': 'LIVER_CANCER', '156': 'GALLBLADDER_CANCER', '157': 'PANCREATIC_CANCER',
    '158': 'PERITONEAL_CANCER', '159': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    '160': 'NASAL_CANCER', '161': 'LARYNGEAL_CANCER',
    '162': 'LUNG_CANCER', '163': 'PLEURAL_CANCER', '164': 'THYMUS_CANCER', 
    '165': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    '170': 'BONE_CANCER', '171': 'SOFT_TISSUE_CANCER', '172': 'MELANOMA',
    '173': 'OTHER_SKIN_CANCER', '174': 'BREAST_CANCER', '175': 'MALE_BREAST_CANCER',
    '176': 'KAPOSI_SARCOMA',
    
    # Genitourinary
    '179': 'UTERINE_CANCER', '180': 'CERVICAL_CANCER', '181': 'PLACENTAL_CANCER',
    '182': 'OVARIAN_CANCER', '183': 'OTHER_FEMALE_GENITAL', '184': 'VULVAR_CANCER',
    '185': 'PROSTATE_CANCER', '186': 'TESTICULAR_CANCER', '187': 'PENILE_CANCER',
    '188': 'BLADDER_CANCER', '189': 'KIDNEY_CANCER',
    
    # Brain and CNS
    '190': 'EYE_CANCER', '191': 'BRAIN_CANCER', '192': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    '193': 'THYROID_CANCER', '194': 'ENDOCRINE_CANCER',
    '195': 'OTHER_CANCER', '196': 'METASTATIC_CANCER', '197': 'SECONDARY_RESPIRATORY',
    '198': 'SECONDARY_DIGESTIVE', '199': 'SECONDARY_CANCER',
}

# Complete ICD-10 to Cancer Type Mapping
ICD10_CANCER_MAPPING = {
    # Head and Neck
    'C00': 'HEAD_NECK_CANCER', 'C01': 'HEAD_NECK_CANCER', 'C02': 'HEAD_NECK_CANCER',
    'C03': 'HEAD_NECK_CANCER', 'C04': 'HEAD_NECK_CANCER', 'C05': 'HEAD_NECK_CANCER',
    'C06': 'HEAD_NECK_CANCER', 'C07': 'HEAD_NECK_CANCER', 'C08': 'HEAD_NECK_CANCER',
    'C09': 'HEAD_NECK_CANCER', 'C10': 'HEAD_NECK_CANCER', 'C11': 'HEAD_NECK_CANCER',
    'C12': 'HEAD_NECK_CANCER', 'C13': 'HEAD_NECK_CANCER', 'C14': 'HEAD_NECK_CANCER',
    
    # Digestive
    'C15': 'ESOPHAGEAL_CANCER', 'C16': 'STOMACH_CANCER', 'C17': 'SMALL_INTESTINE_CANCER',
    'C18': 'COLORECTAL_CANCER', 'C19': 'COLORECTAL_CANCER', 'C20': 'COLORECTAL_CANCER',
    'C21': 'ANAL_CANCER', 'C22': 'LIVER_CANCER', 'C23': 'GALLBLADDER_CANCER',
    'C24': 'BILE_DUCT_CANCER', 'C25': 'PANCREATIC_CANCER', 'C26': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    'C30': 'NASAL_CANCER', 'C31': 'SINUS_CANCER', 'C32': 'LARYNGEAL_CANCER',
    'C33': 'TRACHEAL_CANCER', 'C34': 'LUNG_CANCER', 'C37': 'THYMUS_CANCER',
    'C38': 'HEART_MEDIASTINAL_CANCER', 'C39': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    'C40': 'BONE_CANCER', 'C41': 'BONE_CANCER', 'C43': 'MELANOMA',
    'C44': 'OTHER_SKIN_CANCER', 'C45': 'MESOTHELIOMA', 'C46': 'KAPOSI_SARCOMA',
    'C47': 'PERIPHERAL_NERVE_CANCER', 'C48': 'RETROPERITONEAL_CANCER', 'C49': 'SOFT_TISSUE_CANCER',
    
    # Breast
    'C50': 'BREAST_CANCER',
    
    # Female Genital
    'C51': 'VULVAR_CANCER', 'C52': 'VAGINAL_CANCER', 'C53': 'CERVICAL_CANCER',
    'C54': 'ENDOMETRIAL_CANCER', 'C55': 'UTERINE_CANCER', 'C56': 'OVARIAN_CANCER',
    'C57': 'OTHER_FEMALE_GENITAL', 'C58': 'PLACENTAL_CANCER',
    
    # Male Genital
    'C60': 'PENILE_CANCER', 'C61': 'PROSTATE_CANCER', 'C62': 'TESTICULAR_CANCER',
    'C63': 'OTHER_MALE_GENITAL',
    
    # Urinary
    'C64': 'KIDNEY_CANCER', 'C65': 'RENAL_PELVIS_CANCER', 'C66': 'URETERAL_CANCER',
    'C67': 'BLADDER_CANCER', 'C68': 'OTHER_URINARY_CANCER',
    
    # Brain and CNS
    'C69': 'EYE_CANCER', 'C70': 'MENINGEAL_CANCER', 'C71': 'BRAIN_CANCER',
    'C72': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    'C73': 'THYROID_CANCER', 'C74': 'ADRENAL_CANCER', 'C75': 'OTHER_ENDOCRINE_CANCER',
}

print(f" ICD-9 mapped: {len(ICD9_CANCER_MAPPING)} codes")
print(f" ICD-10 mapped: {len(ICD10_CANCER_MAPPING)} codes")

# =========================================================
# STEP 3: FUNCTION TO MAP ICD TO CANCER TYPE
# =========================================================

def map_icd_to_cancer_type(icd_code, icd_version):
    """Map ICD code to specific cancer type using comprehensive mappings"""
    icd_code = str(icd_code).upper().strip()
    
    if icd_version == 9:
        for code_prefix, cancer_type in ICD9_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    elif icd_version == 10:
        for code_prefix, cancer_type in ICD10_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    
    return 'OTHER_CANCER'

# =========================================================
# STEP 4: HIERARCHICAL COARSE CLASS MAPPING
# =========================================================

# Group similar cancer types for balanced classification
COARSE_CANCER_GROUPS = {
    'LUNG_CANCER': ['LUNG_CANCER', 'TRACHEAL_CANCER'],
    'BREAST_CANCER': ['BREAST_CANCER', 'MALE_BREAST_CANCER'],
    'COLORECTAL_CANCER': ['COLORECTAL_CANCER', 'ANAL_CANCER'],
    'PROSTATE_CANCER': ['PROSTATE_CANCER'],
    'BLADDER_CANCER': ['BLADDER_CANCER'],
    'KIDNEY_CANCER': ['KIDNEY_CANCER', 'RENAL_PELVIS_CANCER'],
    'STOMACH_CANCER': ['STOMACH_CANCER'],
    'LIVER_CANCER': ['LIVER_CANCER', 'BILE_DUCT_CANCER'],
    'PANCREATIC_CANCER': ['PANCREATIC_CANCER'],
    'ESOPHAGEAL_CANCER': ['ESOPHAGEAL_CANCER'],
    'OVARIAN_CANCER': ['OVARIAN_CANCER'],
    'CERVICAL_CANCER': ['CERVICAL_CANCER'],
    'UTERINE_CANCER': ['UTERINE_CANCER', 'ENDOMETRIAL_CANCER'],
    'HEAD_NECK_CANCER': ['HEAD_NECK_CANCER', 'LARYNGEAL_CANCER', 'NASAL_CANCER', 
                         'SINUS_CANCER', 'ORAL_CANCER', 'SALIVARY_CANCER'],
    'THYROID_CANCER': ['THYROID_CANCER'],
    'BRAIN_CANCER': ['BRAIN_CANCER', 'SPINAL_CORD_CANCER', 'MENINGEAL_CANCER'],
    'MELANOMA': ['MELANOMA', 'OTHER_SKIN_CANCER'],
    'KAPOSI_SARCOMA': ['KAPOSI_SARCOMA', 'MESOTHELIOMA'],
    'OTHER_CANCER': ['OTHER_CANCER', 'METASTATIC_CANCER', 'SECONDARY_CANCER',
                     'OTHER_DIGESTIVE_CANCER', 'OTHER_RESPIRATORY_CANCER',
                     'OTHER_FEMALE_GENITAL', 'OTHER_MALE_GENITAL', 'OTHER_URINARY_CANCER',
                     'OTHER_ENDOCRINE_CANCER', 'PERITONEAL_CANCER', 'PLEURAL_CANCER',
                     'THYMUS_CANCER', 'HEART_MEDIASTINAL_CANCER', 'RETROPERITONEAL_CANCER',
                     'PERIPHERAL_NERVE_CANCER', 'SOFT_TISSUE_CANCER', 'BONE_CANCER',
                     'EYE_CANCER', 'ADRENAL_CANCER', 'TESTICULAR_CANCER', 'PENILE_CANCER',
                     'VULVAR_CANCER', 'VAGINAL_CANCER', 'PLACENTAL_CANCER', 'GALLBLADDER_CANCER',
                     'SMALL_INTESTINE_CANCER', 'URETERAL_CANCER']
}

def map_to_coarse_class(cancer_type):
    """Map fine-grained cancer type to coarse category"""
    for coarse_group, fine_types in COARSE_CANCER_GROUPS.items():
        if cancer_type in fine_types:
            return coarse_group
    return 'OTHER_CANCER'

# =========================================================
# STEP 5: HELPER FUNCTION FOR RANDOM EMBEDDINGS
# =========================================================
def maybe_replace_with_random(embeddings, use_random, modality_name, seed=None):
    if not use_random:
        print(f"✓ Using REAL {modality_name} embeddings")
        return embeddings
    
    print(f" Using RANDOM {modality_name} embeddings (seed={seed})")
    if seed is not None:
        np.random.seed(seed)
    
    mean = np.mean(embeddings)
    std = np.std(embeddings)
    random_embs = np.random.normal(mean, std, size=embeddings.shape)
    
    original_norms = np.linalg.norm(embeddings, axis=1)
    random_norms = np.linalg.norm(random_embs, axis=1)
    scale_factors = original_norms / (random_norms + 1e-8)
    random_embs = random_embs * scale_factors[:, np.newaxis]
    
    return random_embs.astype(embeddings.dtype)

# =========================================================
# STEP 6: LOAD AND PROCESS EMBEDDINGS
# =========================================================
print("\n" + "="*60)
print("LOADING AND PROCESSING EMBEDDINGS")
print("="*60)

X_rad = np.load(EMB_RAD_PATH)
hadm_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_subject_ids.npy"))

X_lab = np.load(EMB_LAB_PATH)
hadm_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_subject_ids.npy"))

print(f"\nOriginal shapes:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# Apply random embeddings
X_lab = maybe_replace_with_random(X_lab, USE_RANDOM_LAB, "LAB", RANDOM_SEED)
X_rad = maybe_replace_with_random(X_rad, USE_RANDOM_RAD, "RADIOLOGY", RANDOM_SEED)

# Align admissions
common_hadm_ids = np.intersect1d(hadm_ids_rad, hadm_ids_lab)
idx_rad = np.isin(hadm_ids_rad, common_hadm_ids)
idx_lab = np.isin(hadm_ids_lab, common_hadm_ids)

X_rad = X_rad[idx_rad]
X_lab = X_lab[idx_lab]
hadm_ids = hadm_ids_lab[idx_lab]
subject_ids = subject_ids_lab[idx_lab]

print(f"\nAfter alignment:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# =========================================================
# STEP 7: LOAD DRUG DATA
# =========================================================
print("\n" + "-"*40)
print("LOADING DRUG DATA...")

X_drug = np.load(DRUG_EMB_PATH)
hadm_ids_drug = np.load(DRUG_EMB_PATH.replace(".npy","_hadm_ids.npy"))
drug_sequences = np.load(DRUG_SEQ_PATH)
drug_lengths = np.load(DRUG_SEQ_PATH.replace(".npy","_lengths.npy"))

print(f"Drug embeddings shape: {X_drug.shape}")
print(f"Drug sequences shape: {drug_sequences.shape}")

# Apply random embeddings to drug descriptions
X_drug = maybe_replace_with_random(X_drug, USE_RANDOM_DRUG_DESC, "DRUG DESCRIPTION", RANDOM_SEED)

# =========================================================
# STEP 8: ALIGN ALL DATA
# =========================================================
print("\n" + "-"*40)
print("ALIGNING DATA...")

drug_idx = {hid: i for i, hid in enumerate(hadm_ids_drug)}

aligned_X_drug = []
aligned_drug_sequences = []
aligned_drug_lengths = []
aligned_X_lab = []
aligned_X_rad = []
aligned_subject_ids = []
aligned_hadm_ids = []
missing_drug = 0

for i, hid in enumerate(hadm_ids):
    if hid in drug_idx:
        j = drug_idx[hid]
        aligned_X_drug.append(X_drug[j])
        aligned_drug_sequences.append(drug_sequences[j])
        aligned_drug_lengths.append(drug_lengths[j])
        aligned_X_lab.append(X_lab[i])
        aligned_X_rad.append(X_rad[i])
        aligned_subject_ids.append(subject_ids[i])
        aligned_hadm_ids.append(hid)
    else:
        missing_drug += 1

print(f"Admissions missing drug data: {missing_drug}")

# Stack arrays
X_drug = np.stack(aligned_X_drug)
drug_sequences = np.stack(aligned_drug_sequences)
drug_lengths = np.array(aligned_drug_lengths)
X_lab = np.stack(aligned_X_lab)
X_rad = np.stack(aligned_X_rad)
subject_ids = np.array(aligned_subject_ids)
hadm_ids = np.array(aligned_hadm_ids)

print(f"\nAligned shapes:")
print(f"  Lab: {X_lab.shape}")
print(f"  Rad: {X_rad.shape}")
print(f"  Drug desc: {X_drug.shape}")
print(f"  Drug seq: {drug_sequences.shape}")

# =========================================================
# STEP 9: LOAD AND MAP DIAGNOSIS LABELS (UPDATED)
# =========================================================
print("\n" + "-"*40)
print("LOADING DIAGNOSIS LABELS...")

diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["hadm_id","icd_code","icd_version","seq_num"]
)
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()

# Use EXPANDED cancer filters
mask = (
    ((diag.icd_version==9) & diag.icd_code.str.startswith(EXPANDED_ICD9)) |
    ((diag.icd_version==10) & diag.icd_code.str.startswith(EXPANDED_ICD10))
)

hf_diag = diag[mask]
hadm_to_cancer_type = {}

for hid, g in hf_diag.groupby("hadm_id"):
    primary = g[g.seq_num.isin([1,2])]
    if len(primary) > 0:
        row = primary.sort_values("seq_num").iloc[0]
        cancer_type = map_icd_to_cancer_type(row["icd_code"], row["icd_version"])
        hadm_to_cancer_type[hid] = cancer_type
    else:
        # If no primary, take most common cancer type in admission
        most_common = g['icd_code'].mode()
        if len(most_common) > 0:
            cancer_type = map_icd_to_cancer_type(most_common[0], g.iloc[0]['icd_version'])
            hadm_to_cancer_type[hid] = cancer_type
        else:
            hadm_to_cancer_type[hid] = "OTHER_CANCER"

# Apply diagnosis filter
final_mask = np.array([hid in hadm_to_cancer_type for hid in hadm_ids])
X_lab = X_lab[final_mask]
X_rad = X_rad[final_mask]
X_drug = X_drug[final_mask]
drug_sequences = drug_sequences[final_mask]
drug_lengths = drug_lengths[final_mask]
subject_ids = subject_ids[final_mask]
hadm_ids = hadm_ids[final_mask]

y_hf_fine = np.array([hadm_to_cancer_type[hid] for hid in hadm_ids])

print(f"After diagnosis filter: {len(y_hf_fine)} admissions")

# =========================================================
# STEP 10: COLLAPSE VERY RARE CLASSES
# =========================================================
print("\n" + "-"*40)
print("COLLAPSING VERY RARE CLASSES...")

MIN_SAMPLES = 100
counts = pd.Series(y_hf_fine).value_counts()
rare = counts[counts < MIN_SAMPLES].index

y_hf_fine = np.array([
    "RARE_CANCER" if lbl in rare else lbl
    for lbl in y_hf_fine
])

print("\nFine-grained label distribution:")
label_counts = pd.Series(y_hf_fine).value_counts()
for label, count in label_counts.items():
    pct = count / len(y_hf_fine) * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 11: APPLY HIERARCHICAL MAPPING
# =========================================================
print("\n" + "-"*40)
print("APPLYING HIERARCHICAL MAPPING...")

if USE_HIERARCHICAL:
    y_hf = np.array([map_to_coarse_class(lbl) for lbl in y_hf_fine])
    print("\n Using HIERARCHICAL coarse classes:")
else:
    y_hf = y_hf_fine.copy()
    print("\n Using FINE-grained original classes:")

label_counts = pd.Series(y_hf).value_counts()
total = len(y_hf)
print(f"\nFinal label distribution ({len(label_counts)} classes):")
for label, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 12: AGGREGATE BY PATIENT
# =========================================================
print("\n" + "-"*40)
print("AGGREGATING BY PATIENT...")

patient_to_lab = defaultdict(list)
patient_to_rad = defaultdict(list)
patient_to_drug_desc = defaultdict(list)
patient_to_labels = defaultdict(list)
patient_to_drugs = defaultdict(list)

for lab, rad, drug_desc, lbl, drug_seq, pid in zip(
    X_lab, X_rad, X_drug, y_hf, drug_sequences, subject_ids
):
    patient_to_lab[pid].append(lab)
    patient_to_rad[pid].append(rad)
    patient_to_drug_desc[pid].append(drug_desc)
    patient_to_labels[pid].append(lbl)
    patient_to_drugs[pid].append(drug_seq)

lab_sequences_by_patient = [np.stack(v) for v in patient_to_lab.values()]
rad_sequences_by_patient = [np.stack(v) for v in patient_to_rad.values()]
drug_desc_sequences_by_patient = [np.stack(v) for v in patient_to_drug_desc.values()]
drug_labels_by_patient = [np.stack(v) for v in patient_to_drugs.values()]
labels_by_patient = [np.array(v) for v in patient_to_labels.values()]
patient_ids = list(patient_to_lab.keys())

print(f"\nFinal dataset:")
print(f"  Patients: {len(patient_ids)}")
print(f"  Total admissions: {sum(len(seq) for seq in labels_by_patient)}")

# =========================================================
# STEP 13: LOAD DDI MATRIX
# =========================================================
print("\n" + "-"*40)
print("LOADING DDI MATRIX...")

with open(DDI_PATH, "rb") as f:
    mapped_ddi_pairs = pickle.load(f)

with open(DRUG2IDX_PATH, "rb") as f:
    drug2idx = pickle.load(f)

def normalize_drug_name(name):
    if name is None:
        return ""
    name = name.lower().strip()
    if name.startswith("*nf*"):
        name = name[4:].strip()
    return name

norm_drug2idx = {normalize_drug_name(d): idx for d, idx in drug2idx.items()}
n_drugs = len(norm_drug2idx)
print(f"Number of drugs: {n_drugs}")

severity_weight = {"minor": 0.5, "moderate": 2.0, "major": 5.0}

def build_ddi_severity_matrices(ddi_pairs, drug2idx, n_drugs, severity_weight):
    ddi_matrix = np.zeros((n_drugs, n_drugs), dtype=np.float32)
    for drug1, drug2, severity in ddi_pairs:
        if drug1 in drug2idx and drug2 in drug2idx:
            i, j = drug2idx[drug1], drug2idx[drug2]
            weight = severity_weight.get(severity, 1.0)
            ddi_matrix[i, j] = weight
            ddi_matrix[j, i] = weight
    return ddi_matrix

ddi_severity_matrix = build_ddi_severity_matrices(
    ddi_pairs=mapped_ddi_pairs,
    drug2idx=norm_drug2idx,
    n_drugs=n_drugs,
    severity_weight=severity_weight
)

print(f"DDI severity matrix shape: {ddi_severity_matrix.shape}")

# =========================================================
# STEP 14: CREATE LABEL ENCODER FOR TRAINING
# =========================================================
print("\n" + "-"*40)
print("CREATING LABEL ENCODER...")

le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(seq) for seq in labels_by_patient]
n_classes = len(le_hf.classes_)

print(f"Number of classes for training: {n_classes}")
print(f"Classes: {list(le_hf.classes_)}")

# =========================================================
# STEP 15: FINAL VALIDATION
# =========================================================
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

n = len(hadm_ids)
assert X_lab.shape[0] == n, f"Lab shape mismatch"
assert X_rad.shape[0] == n, f"Rad shape mismatch"
assert X_drug.shape[0] == n, f"Drug desc shape mismatch"

print("\n PREPROCESSING COMPLETE!")
print(f"\n Ablation configuration:")
print(f"   - LAB: {'RANDOM' if USE_RANDOM_LAB else 'REAL'}")
print(f"   - RADIOLOGY: {'RANDOM' if USE_RANDOM_RAD else 'REAL'}")
print(f"   - DRUG DESCRIPTION: {'RANDOM' if USE_RANDOM_DRUG_DESC else 'REAL'}")
print(f"   - HIERARCHICAL CLASSES: {'YES' if USE_HIERARCHICAL else 'NO'}")

print(f"\n Dataset statistics:")
print(f"   - Patients: {len(patient_ids)}")
print(f"   - Admissions: {n}")
print(f"   - Classes: {n_classes}")
print(f"   - Drugs: {n_drugs}")

print(f"\n Final class distribution:")
for cls, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {cls:30s}: {count:5d} ({pct:5.1f}%) {bar}")


EXPANDED CANCER CLASSIFICATION + ABLATION STUDY

----------------------------------------
DEFINING EXPANDED CANCER TYPE MAPPING...
✅ Expanded ICD-9 prefixes: 54
✅ Expanded ICD-10 prefixes: 69
✅ ICD-9 mapped: 54 codes
✅ ICD-10 mapped: 69 codes

LOADING AND PROCESSING EMBEDDINGS

Original shapes:
  Radiology: (12708, 2560)
  Lab: (19866, 2560)
✓ Using REAL LAB embeddings
✓ Using REAL RADIOLOGY embeddings

After alignment:
  Radiology: (12093, 2560)
  Lab: (12093, 2560)

----------------------------------------
LOADING DRUG DATA...
Drug embeddings shape: (21158, 2560)
Drug sequences shape: (21173, 689)
✓ Using REAL DRUG DESCRIPTION embeddings

----------------------------------------
ALIGNING DATA...
Admissions missing drug data: 41

Aligned shapes:
  Lab: (12052, 2560)
  Rad: (12052, 2560)
  Drug desc: (12052, 2560)
  Drug seq: (12052, 689)

----------------------------------------
LOADING DIAGNOSIS LABELS...
After diagnosis filter: 12052 admissions

------------------------------------

FusionMed with Transformer

In [6]:
# ============================================================
#  MULTI-MODAL CAUSAL TRANSFORMER 
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# -----------------------------
# Device & settings
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32
hidden_dim = 256
num_layers = 2
num_heads = 4
dropout = 0.1
epochs = 50
learning_rate = 1e-5
weight_decay = 1e-4
top_k = 20

# Loss weights
alpha = 0.2  # Classification weight
beta = 0.7   # Drug prediction weight
gamma = 0.1  # DDI penalty weight
lambda_sparse = 1e-7
bpr_temperature = 1.0

# Mixed precision training
use_amp = True
if use_amp and device == "cuda":
    scaler = torch.amp.GradScaler('cuda')
else:
    use_amp = False
    scaler = None

# -----------------------------
# Encode HF labels
# -----------------------------
le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(lbl_seq) for lbl_seq in labels_by_patient]
n_classes = len(le_hf.classes_)
print(f"Number of HF classes: {n_classes}")

# -----------------------------
# Drugs
# -----------------------------
n_drugs = ddi_severity_matrix.shape[0]
print(f"Detected n_drugs: {n_drugs}")

# -----------------------------
# Multi-hot encode drug labels (targets for next admission)
# -----------------------------
drug_labels_by_patient_enc = []
for seq in drug_labels_by_patient:
    seq_enc = np.zeros((len(seq), n_drugs), dtype=np.float32)
    for t, drugs in enumerate(seq):
        if len(drugs) > 0:
            drugs_valid = [d for d in drugs if d < n_drugs]
            if drugs_valid:
                seq_enc[t, drugs_valid] = 1.0
    drug_labels_by_patient_enc.append(seq_enc)

# -----------------------------
# Precompute DDI matrix
# -----------------------------
if torch.is_tensor(ddi_severity_matrix):
    ddi_matrix_np = ddi_severity_matrix.cpu().numpy()
else:
    ddi_matrix_np = ddi_severity_matrix

ddi_matrix_norm = ddi_matrix_np / (ddi_matrix_np.max() + 1e-8)
ddi_matrix_tensor = torch.tensor(ddi_matrix_norm, dtype=torch.float32, device=device)

# -----------------------------
# Dataset (Separate modalities)
# -----------------------------
class MultiModalDataset(torch.utils.data.Dataset):
    def __init__(self, lab_sequences, rad_sequences, drug_desc_sequences, 
                 drug_history_sequences, class_labels, drug_targets):
        self.lab_sequences = lab_sequences
        self.rad_sequences = rad_sequences
        self.drug_desc_sequences = drug_desc_sequences
        self.drug_history_sequences = drug_history_sequences
        self.class_labels = class_labels
        self.drug_targets = drug_targets
        
    def __len__(self):
        return len(self.lab_sequences)
    
    def __getitem__(self, idx):
        # Get sequences
        lab_seq = self.lab_sequences[idx]
        rad_seq = self.rad_sequences[idx]
        drug_desc_seq = self.drug_desc_sequences[idx]
        drug_history_seq = self.drug_history_sequences[idx]
        
        # Ensure drug_history is 1D
        if isinstance(drug_history_seq, np.ndarray):
            if drug_history_seq.ndim == 2:
                if drug_history_seq.shape[1] == 1:
                    drug_history_seq = drug_history_seq.flatten()
                elif drug_history_seq.shape[0] == 1:
                    drug_history_seq = drug_history_seq.flatten()
        
        # Make sure all sequences have same length
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq))
        
        lab_seq = lab_seq[:min_len]
        rad_seq = rad_seq[:min_len]
        drug_desc_seq = drug_desc_seq[:min_len]
        drug_history_seq = drug_history_seq[:min_len]
        
        return (torch.tensor(lab_seq, dtype=torch.float32),
                torch.tensor(rad_seq, dtype=torch.float32),
                torch.tensor(drug_desc_seq, dtype=torch.float32),
                torch.tensor(drug_history_seq, dtype=torch.long),
                torch.tensor(self.class_labels[idx][:min_len], dtype=torch.long),
                torch.tensor(self.drug_targets[idx][:min_len], dtype=torch.float32))

def collate_fn(batch):
    lab_seq, rad_seq, drug_desc, drug_history, ys_class, ys_drug = zip(*batch)
    
    # Get lengths
    lengths = torch.tensor([len(x) for x in lab_seq], dtype=torch.long)
    
    # Pad sequences
    lab_padded = pad_sequence(lab_seq, batch_first=True, padding_value=0.0)
    rad_padded = pad_sequence(rad_seq, batch_first=True, padding_value=0.0)
    drug_desc_padded = pad_sequence(drug_desc, batch_first=True, padding_value=0.0)
    drug_history_padded = pad_sequence(drug_history, batch_first=True, padding_value=0)
    cls_padded = pad_sequence(ys_class, batch_first=True, padding_value=-100)
    drug_padded = pad_sequence(ys_drug, batch_first=True, padding_value=0.0)
    
    return (lab_padded, rad_padded, drug_desc_padded, drug_history_padded,
            cls_padded, drug_padded, lengths)

# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term[:pe[:, 0::2].shape[1]])
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# -----------------------------
# Causal Mask Generator (Autoregressive)
# -----------------------------
def generate_causal_mask(seq_len, device):
    """
    Creates causal mask to prevent attending to future positions.
    This makes the transformer autoregressive (like GPT, not BERT).
    """
    # Upper triangular matrix (excluding diagonal) = future positions
    return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()

# -----------------------------
# VECTORIZED BPR LOSS
# -----------------------------
def bpr_loss_vectorized(scores, positive_mask, temperature=1.0):
    """
    Numerically stable and fully vectorized Multi-Label BPR Loss.
    """
    # 1. Sum of positive scores per sample
    pos_count = positive_mask.sum(dim=1).clamp(min=1)
    pos_scores_mean = (scores * positive_mask).sum(dim=1) / pos_count
    
    # 2. Pairwise difference (mean of positives minus all scores)
    diff = (pos_scores_mean.unsqueeze(1) - scores) / temperature
    
    # 3. Apply log-sigmoid safely
    logsigs = F.logsigmoid(diff)
    
    # 4. Mask out positive items (we only want the loss for negative items)
    neg_mask = 1 - positive_mask
    masked_logsigs = logsigs * neg_mask
    
    # 5. Average loss over negative items ONLY
    neg_count = neg_mask.sum(dim=1).clamp(min=1)
    loss_per_sample = -masked_logsigs.sum(dim=1) / neg_count
    
    return loss_per_sample.mean()

bpr_loss = bpr_loss_vectorized

# -----------------------------
# SIMPLIFIED CAUSAL TRANSFORMER (Fixed - No Time Bias)
# -----------------------------
class MultiModalCausalTransformer(nn.Module):
    def __init__(self, lab_dim, rad_dim, drug_desc_dim, n_drugs, n_classes,
                 hidden_dim=256, num_layers=2, num_heads=4, dropout=0.1):
        super().__init__()
        
        # Patient state encoders (LAB + RAD)
        self.lab_proj = nn.Sequential(
            nn.Linear(lab_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.rad_proj = nn.Sequential(
            nn.Linear(rad_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Drug description encoder
        self.drug_desc_proj = nn.Sequential(
            nn.Linear(drug_desc_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Drug history encoder (shifted to prevent leakage)
        self.drug_history_embed = nn.Embedding(n_drugs, hidden_dim)
        
        # Cross-modal attention (drug-aware patient representation)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        
        # Gated fusion
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim * 4),
            nn.Sigmoid()
        )
        
        self.fusion_proj = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(hidden_dim)
        
        # Causal Transformer Encoder (autoregressive)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Layer norm after transformer
        self.transformer_norm = nn.LayerNorm(hidden_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Classification head
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes)
        )
        
        # Drug prediction head
        self.drug_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_drugs)
        )
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight, gain=0.5)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0.0)
        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)
                    
    def forward(self, lab_x, rad_x, drug_desc_x, drug_history_x, lengths):
        # Project modalities
        lab_h = self.lab_proj(lab_x)
        rad_h = self.rad_proj(rad_x)
        drug_desc_h = self.drug_desc_proj(drug_desc_x)
        
        # FIX: Shift drug history to prevent future leakage
        # At time t, we only have access to drugs from previous admissions (t-1)
        # For t=0, we use zeros (no previous drugs)
        drug_history_shifted = torch.zeros_like(drug_history_x)
        if drug_history_x.shape[1] > 1:
            drug_history_shifted[:, 1:] = drug_history_x[:, :-1]  # Shift right by 1
        drug_history_h = self.drug_history_embed(drug_history_shifted)
        
        # Combine patient state
        patient_state = (lab_h + rad_h) / 2
        
        # Cross-modal attention (drug-aware patient representation)
        attended_drugs, _ = self.cross_attention(
            query=patient_state,
            key=drug_desc_h,
            value=drug_desc_h
        )
        
        # Concatenate and fuse all modalities
        combined = torch.cat([patient_state, attended_drugs, drug_desc_h, drug_history_h], dim=-1)
        gate = self.fusion_gate(combined)
        gated = combined * gate
        fused = self.fusion_proj(gated)
        
        # Add positional encoding
        fused = self.pos_encoder(fused)
        
        # Generate causal mask (autoregressive - cannot see future)
        seq_len = fused.shape[1]
        causal_mask = generate_causal_mask(seq_len, fused.device)
        
        # Padding mask
        padding_mask = torch.arange(fused.shape[1], device=lengths.device).expand(len(lengths), fused.shape[1]) >= lengths.unsqueeze(1)
        
        # Causal Transformer forward pass
        transformer_out = self.transformer(
            fused, 
            mask=causal_mask, 
            src_key_padding_mask=padding_mask
        )
        
        # Apply layer norm and dropout
        out = self.transformer_norm(transformer_out)
        out = self.dropout(out)
        
        # Output heads
        cls_logits = torch.clamp(self.cls_head(out), -10, 10)
        drug_logits = torch.clamp(self.drug_head(out), -5, 5)
        
        return cls_logits, drug_logits

# -----------------------------
# Vectorized DDI Penalty
# -----------------------------
def fast_ddi_penalty(drug_logits, lengths):
    B, T, D = drug_logits.shape
    
    probs = torch.sigmoid(torch.clamp(drug_logits, -10, 10))
    mask = torch.arange(T, device=drug_logits.device).expand(B, T) < lengths.unsqueeze(1)
    mask = mask.float()
    
    probs_flat = probs.reshape(-1, D)
    ddi_contrib = (probs_flat @ ddi_matrix_tensor) * probs_flat
    ddi_per_step = ddi_contrib.sum(dim=1).reshape(B, T)
    
    ddi_masked = ddi_per_step * mask
    valid_counts = mask.sum(dim=1).clamp(min=1)
    penalty = (ddi_masked.sum(dim=1) / valid_counts).mean()
    
    return penalty * 0.0005

# -----------------------------
# Loss Function
# -----------------------------
def compute_loss(drug_scores, y_drug, lengths, cls_logits, y_cls,
                 alpha_cls=1.0, beta_drug=1.0, gamma_ddi=0.0, lambda_sparse=1e-7):
    
    # Vectorized BPR Loss (evaluated per time step)
    B, T, D = drug_scores.shape
    bpr_losses = []
    
    for t in range(T):
        if t >= lengths.max():
            break
        scores_t = drug_scores[:, t, :]
        targets_t = y_drug[:, t, :]
        
        bpr_t = bpr_loss(scores_t, targets_t, temperature=bpr_temperature)
        if not torch.isnan(bpr_t) and not torch.isinf(bpr_t):
            bpr_losses.append(bpr_t)
    
    if len(bpr_losses) > 0:
        bpr = torch.stack(bpr_losses).mean()
    else:
        bpr = torch.tensor(0.0, device=drug_scores.device)
    
    # DDI penalty
    ddi = fast_ddi_penalty(drug_scores, lengths) if gamma_ddi > 0 else torch.tensor(0.0, device=drug_scores.device)
    
    # L1 sparsity penalty
    probs = torch.sigmoid(torch.clamp(drug_scores, -10, 10))
    l1 = torch.mean(torch.abs(probs))
    l1 = torch.nan_to_num(l1)
    
    # Classification loss
    cls_loss = F.cross_entropy(
        cls_logits[:, :-1].reshape(-1, n_classes),
        y_cls[:, 1:].reshape(-1),
        ignore_index=-100,
        label_smoothing=0.0
    )
    
    total_loss = (alpha_cls * cls_loss + 
                  beta_drug * bpr + 
                  gamma_ddi * ddi + 
                  lambda_sparse * l1)
    
    total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=1e3, neginf=1e3)
    
    return total_loss, bpr.item(), ddi.item(), l1.item(), cls_loss.item()

# -----------------------------
# Training Function
# -----------------------------
def train_epoch(model, loader, optimizer, scaler=None):
    model.train()
    total_loss = 0
    total_bpr = 0
    total_ddi = 0
    total_l1 = 0
    total_cls = 0
    n_batches = 0
    
    for lab_x, rad_x, drug_desc, drug_history, y_cls, y_drug, lengths in loader:
        lab_x = lab_x.to(device, non_blocking=True)
        rad_x = rad_x.to(device, non_blocking=True)
        drug_desc = drug_desc.to(device, non_blocking=True)
        drug_history = drug_history.to(device, non_blocking=True)
        y_cls = y_cls.to(device, non_blocking=True)
        y_drug = y_drug.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                cls_logits, drug_logits = model(lab_x, rad_x, drug_desc, drug_history, lengths)
                loss, bpr, ddi, l1, cls = compute_loss(
                    drug_logits[:, :-1], y_drug[:, 1:], lengths - 1,
                    cls_logits, y_cls,
                    alpha_cls=alpha, beta_drug=beta, gamma_ddi=gamma, lambda_sparse=lambda_sparse
                )
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            cls_logits, drug_logits = model(lab_x, rad_x, drug_desc, drug_history, lengths)
            loss, bpr, ddi, l1, cls = compute_loss(
                drug_logits[:, :-1], y_drug[:, 1:], lengths - 1,
                cls_logits, y_cls,
                alpha_cls=alpha, beta_drug=beta, gamma_ddi=gamma, lambda_sparse=lambda_sparse
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        total_loss += loss.item()
        total_bpr += bpr
        total_ddi += ddi
        total_l1 += l1
        total_cls += cls
        n_batches += 1
    
    return (total_loss / n_batches, total_bpr / n_batches, total_ddi / n_batches,
            total_l1 / n_batches, total_cls / n_batches)

# -----------------------------
# Main Cross-validation Loop
# -----------------------------
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
strat_labels = [lbl_seq[0] for lbl_seq in labels_by_patient_enc]

micro_f1_scores = []
macro_f1_scores = []
hf_roc_auc_scores = []
recall_scores = []
ndcg_scores = []
ddi_rate_scores = []
ddi_severity_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(patient_ids, strat_labels), 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/10")
    print(f"{'='*60}")
    
    # Split train/val
    train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42)
    
    # Get dimensions
    lab_dim = lab_sequences_by_patient[0].shape[-1]
    rad_dim = rad_sequences_by_patient[0].shape[-1]
    drug_desc_dim = X_drug.shape[-1]
    
    print(f"LAB dimension: {lab_dim}")
    print(f"RAD dimension: {rad_dim}")
    print(f"Drug description dimension: {drug_desc_dim}")
    
    # Create datasets with all modalities
    train_lab = []
    train_rad = []
    train_drug_desc = []
    train_drug_history = []
    train_labels = []
    train_targets = []
    
    for i in train_idx:
        pid = patient_ids[i]
        patient_mask = subject_ids == pid
        
        lab_seq = lab_sequences_by_patient[i]
        rad_seq = rad_sequences_by_patient[i]
        drug_desc_seq = X_drug[patient_mask]
        drug_history_seq = drug_sequences[patient_mask]
        
        if drug_history_seq.ndim == 2:
            drug_history_seq = drug_history_seq.flatten()
        
        label_seq = labels_by_patient_enc[i]
        target_seq = drug_labels_by_patient_enc[i]
        
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq), len(label_seq), len(target_seq))
        
        train_lab.append(lab_seq[:min_len])
        train_rad.append(rad_seq[:min_len])
        train_drug_desc.append(drug_desc_seq[:min_len])
        train_drug_history.append(drug_history_seq[:min_len])
        train_labels.append(label_seq[:min_len])
        train_targets.append(target_seq[:min_len])
    
    val_lab = []
    val_rad = []
    val_drug_desc = []
    val_drug_history = []
    val_labels = []
    val_targets = []
    
    for i in val_idx:
        pid = patient_ids[i]
        patient_mask = subject_ids == pid
        
        lab_seq = lab_sequences_by_patient[i]
        rad_seq = rad_sequences_by_patient[i]
        drug_desc_seq = X_drug[patient_mask]
        drug_history_seq = drug_sequences[patient_mask]
        
        if drug_history_seq.ndim == 2:
            drug_history_seq = drug_history_seq.flatten()
        
        label_seq = labels_by_patient_enc[i]
        target_seq = drug_labels_by_patient_enc[i]
        
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq), len(label_seq), len(target_seq))
        
        val_lab.append(lab_seq[:min_len])
        val_rad.append(rad_seq[:min_len])
        val_drug_desc.append(drug_desc_seq[:min_len])
        val_drug_history.append(drug_history_seq[:min_len])
        val_labels.append(label_seq[:min_len])
        val_targets.append(target_seq[:min_len])
    
    test_lab = []
    test_rad = []
    test_drug_desc = []
    test_drug_history = []
    test_labels = []
    test_targets = []
    
    for i in test_idx:
        pid = patient_ids[i]
        patient_mask = subject_ids == pid
        
        lab_seq = lab_sequences_by_patient[i]
        rad_seq = rad_sequences_by_patient[i]
        drug_desc_seq = X_drug[patient_mask]
        drug_history_seq = drug_sequences[patient_mask]
        
        if drug_history_seq.ndim == 2:
            drug_history_seq = drug_history_seq.flatten()
        
        label_seq = labels_by_patient_enc[i]
        target_seq = drug_labels_by_patient_enc[i]
        
        min_len = min(len(lab_seq), len(rad_seq), len(drug_desc_seq), len(drug_history_seq), len(label_seq), len(target_seq))
        
        test_lab.append(lab_seq[:min_len])
        test_rad.append(rad_seq[:min_len])
        test_drug_desc.append(drug_desc_seq[:min_len])
        test_drug_history.append(drug_history_seq[:min_len])
        test_labels.append(label_seq[:min_len])
        test_targets.append(target_seq[:min_len])
    
    train_dataset = MultiModalDataset(
        train_lab, train_rad, train_drug_desc, train_drug_history,
        train_labels, train_targets
    )
    
    val_dataset = MultiModalDataset(
        val_lab, val_rad, val_drug_desc, val_drug_history,
        val_labels, val_targets
    )
    
    test_dataset = MultiModalDataset(
        test_lab, test_rad, test_drug_desc, test_drug_history,
        test_labels, test_targets
    )
    
    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                            collate_fn=collate_fn, num_workers=0, pin_memory=True)
    
    # Model - Causal Transformer (Fixed version)
    model = MultiModalCausalTransformer(
        lab_dim=lab_dim,
        rad_dim=rad_dim,
        drug_desc_dim=drug_desc_dim,
        n_drugs=n_drugs,
        n_classes=n_classes,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_heads=num_heads,
        dropout=dropout
    ).to(device)
    
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Training
    best_val_loss = float('inf')
    patience = 0
    
    for epoch in range(epochs):
        train_loss, bpr, ddi, l1, cls = train_epoch(model, train_loader, optimizer, scaler)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for lab_x, rad_x, drug_desc, drug_history, y_cls, y_drug, lengths in val_loader:
                lab_x = lab_x.to(device, non_blocking=True)
                rad_x = rad_x.to(device, non_blocking=True)
                drug_desc = drug_desc.to(device, non_blocking=True)
                drug_history = drug_history.to(device, non_blocking=True)
                y_cls = y_cls.to(device, non_blocking=True)
                y_drug = y_drug.to(device, non_blocking=True)
                lengths = lengths.to(device, non_blocking=True)
                
                cls_logits, drug_logits = model(lab_x, rad_x, drug_desc, drug_history, lengths)
                loss, _, _, _, _ = compute_loss(
                    drug_logits[:, :-1], y_drug[:, 1:], lengths - 1,
                    cls_logits, y_cls,
                    alpha_cls=alpha, beta_drug=beta, gamma_ddi=gamma, lambda_sparse=lambda_sparse
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {train_loss:.4f} | Val: {val_loss:.4f}")
        print(f"        CLS: {cls:.4f}, BPR: {bpr:.4f}, DDI: {ddi:.4f}, L1: {l1:.6f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            torch.save(model.state_dict(), f'best_model_fold{fold}_causal_transformer.pt')
        else:
            patience += 1
            if patience >= 7:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(torch.load(f'best_model_fold{fold}_causal_transformer.pt', weights_only=False))
    model.eval()
    
    # Evaluation
    y_true_cls, y_pred_cls, y_proba_cls = [], [], []
    y_true_drug, y_score_drug = [], []
    
    with torch.no_grad():
        for lab_x, rad_x, drug_desc, drug_history, y_cls, y_drug, lengths in test_loader:
            lab_x = lab_x.to(device, non_blocking=True)
            rad_x = rad_x.to(device, non_blocking=True)
            drug_desc = drug_desc.to(device, non_blocking=True)
            drug_history = drug_history.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            
            cls_logits, drug_logits = model(lab_x, rad_x, drug_desc, drug_history, lengths)
            cls_probs = F.softmax(cls_logits, -1)
            
            for i in range(len(lengths)):
                L = lengths[i].item()
                if L <= 1: continue
                    
                y_true_cls.extend(y_cls[i, 1:L].cpu().numpy())
                y_pred_cls.extend(cls_probs[i, :-1][:L-1].argmax(-1).cpu().numpy())
                y_proba_cls.extend(cls_probs[i, :-1][:L-1].cpu().numpy())
                y_true_drug.append(y_drug[i, 1:L].cpu().numpy())
                y_score_drug.append(torch.sigmoid(drug_logits[i, :-1][:L-1]).cpu().numpy())
    
    # Convert to numpy arrays
    y_true_drug_arr = np.vstack(y_true_drug) if y_true_drug else np.array([])
    y_score_drug_arr = np.vstack(y_score_drug) if y_score_drug else np.array([])
    
    # Classification metrics
    if y_true_cls:
        y_true_cls = np.array(y_true_cls)
        y_pred_cls = np.array(y_pred_cls)
        y_proba_cls = np.array(y_proba_cls)
        
        micro_f1 = f1_score(y_true_cls, y_pred_cls, average="micro")
        macro_f1 = f1_score(y_true_cls, y_pred_cls, average="macro")
        
        try:
            roc_auc = roc_auc_score(y_true_cls, y_proba_cls, multi_class="ovr", average="macro")
        except:
            roc_auc = 0.0
    else:
        micro_f1 = macro_f1 = roc_auc = 0.0
    
    # Drug metrics
    if len(y_true_drug_arr) > 0:
        recall_array = recall_at_k(y_true_drug_arr, y_score_drug_arr, top_k)
        recall_val = float(np.mean(recall_array))
        
        ndcg_val = float(ndcg_at_k(y_true_drug_arr, y_score_drug_arr, top_k))
        
        ddi_rate_val, ddi_sev_avg = ddi_rate(y_score_drug_arr, ddi_matrix_np, top_k)
        ddi_rate_val = float(ddi_rate_val)
        ddi_sev_avg = float(ddi_sev_avg)
    else:
        recall_val = ndcg_val = ddi_rate_val = ddi_sev_avg = 0.0
    
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)
    hf_roc_auc_scores.append(roc_auc)
    recall_scores.append(recall_val)
    ndcg_scores.append(ndcg_val)
    ddi_rate_scores.append(ddi_rate_val)
    ddi_severity_scores.append(ddi_sev_avg)
    
    print(f"\nFold {fold} Results:")
    print(f"  HF - Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f} | ROC-AUC: {roc_auc:.4f}")
    print(f"  Drugs - Recall@{top_k}: {recall_val:.4f} | NDCG@{top_k}: {ndcg_val:.4f}")
    print(f"  Safety - DDI rate: {ddi_rate_val:.4f} | Avg severity: {ddi_sev_avg:.4f}")

# Summary
if micro_f1_scores:
    print("\n" + "="*60)
    print("10-Fold CV Summary - Multi-Modal Causal Transformer")
    print("="*60)
    print(f"Micro-F1         : {np.mean(micro_f1_scores):.4f} ± {np.std(micro_f1_scores):.4f}")
    print(f"Macro-F1         : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    print(f"ROC-AUC          : {np.mean(hf_roc_auc_scores):.4f} ± {np.std(hf_roc_auc_scores):.4f}")
    print(f"Recall@{top_k}   : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
    print(f"NDCG@{top_k}     : {np.mean(ndcg_scores):.4f} ± {np.std(ndcg_scores):.4f}")
    print(f"DDI rate         : {np.mean(ddi_rate_scores):.4f} ± {np.std(ddi_rate_scores):.4f}")
    print(f"Avg DDI severity : {np.mean(ddi_severity_scores):.4f} ± {np.std(ddi_severity_scores):.4f}")

Number of HF classes: 7
Detected n_drugs: 2374

Fold 1/10
LAB dimension: 2560
RAD dimension: 2560
Drug description dimension: 2560
Trainable parameters: 6,104,781
Epoch  1/50 | Loss: 0.9062 | Val: 0.8563
        CLS: 1.6733, BPR: 0.6675, DDI: 1.0427, L1: 0.486835
Epoch  2/50 | Loss: 0.7871 | Val: 0.6916
        CLS: 1.6228, BPR: 0.5469, DDI: 0.7967, L1: 0.419712
Epoch  3/50 | Loss: 0.6237 | Val: 0.5408
        CLS: 1.6132, BPR: 0.3699, DDI: 0.4210, L1: 0.305737
Epoch  4/50 | Loss: 0.4993 | Val: 0.4391
        CLS: 1.5719, BPR: 0.2367, DDI: 0.1924, L1: 0.205189
Epoch  5/50 | Loss: 0.4215 | Val: 0.3893
        CLS: 1.4427, BPR: 0.1732, DDI: 0.1171, L1: 0.151427
Epoch  6/50 | Loss: 0.3836 | Val: 0.3603
        CLS: 1.3781, BPR: 0.1421, DDI: 0.0850, L1: 0.123591
Epoch  7/50 | Loss: 0.3592 | Val: 0.3489
        CLS: 1.3383, BPR: 0.1215, DDI: 0.0654, L1: 0.104597
Epoch  8/50 | Loss: 0.3406 | Val: 0.3242
        CLS: 1.2957, BPR: 0.1085, DDI: 0.0552, L1: 0.092314
Epoch  9/50 | Loss: 0.3250 | 